# Tokenizadores

In [1138]:
from tokenizers import Tokenizer
import torch
import sys

In [1139]:
tokenizador = Tokenizer.from_file("./data/tokenizer.json")
texto = "O gato dorme"

tokens = [5, 8, 2, 9, 4, 7]
context_size = 3
stride = 1

In [1140]:
resultado = tokenizador.encode(texto)
# ['O', 'Ġga', 'to', 'Ġdor', 'me'] => Ġ representa espaço
print(resultado.tokens)
print(resultado.ids)

['O', 'Ġga', 'to', 'Ġdor', 'me']
[47, 4326, 287, 4956, 547]


In [1141]:
texto_reconstruido = tokenizador.decode(resultado.ids)
print(texto_reconstruido)
print(texto_reconstruido == texto)

O gato dorme
True


# Ids

In [1142]:
entrada = resultado.ids[:-1]
alvo = resultado.ids[1:]

print(entrada, alvo)

[47, 4326, 287, 4956] [4326, 287, 4956, 547]


# Stride

In [1143]:
for i in range(0, len(tokens) - context_size, stride):
    print("i = ", i)
    x = tokens[i: i + context_size]
    y = tokens[i + 1: i + context_size + 1]
    print(x, y)

i =  0
[5, 8, 2] [8, 2, 9]
i =  1
[8, 2, 9] [2, 9, 4]
i =  2
[2, 9, 4] [9, 4, 7]


# Tensores

In [1144]:
x = []
y = []

for i in range(0, len(tokens) - context_size, stride):
    print("i = ", i)
    x.append(tokens[i: i + context_size])
    y.append(tokens[i + 1: i + context_size + 1])
    print(x, y)
    
tensor_x = torch.tensor(x, dtype=torch.long)
tensor_y = torch.tensor(y, dtype=torch.long)
print(tensor_x, tensor_y)
print(tensor_x.shape)
print(tensor_y.shape)

i =  0
[[5, 8, 2]] [[8, 2, 9]]
i =  1
[[5, 8, 2], [8, 2, 9]] [[8, 2, 9], [2, 9, 4]]
i =  2
[[5, 8, 2], [8, 2, 9], [2, 9, 4]] [[8, 2, 9], [2, 9, 4], [9, 4, 7]]
tensor([[5, 8, 2],
        [8, 2, 9],
        [2, 9, 4]]) tensor([[8, 2, 9],
        [2, 9, 4],
        [9, 4, 7]])
torch.Size([3, 3])
torch.Size([3, 3])


# Batch

In [1145]:
batch_size = 2


def batching(tensor_list, size=batch_size):
    batch = []
    for i in range(0, len(tensor_list), size):
        batch.append(tensor_list[i : i + size])

    return batch


batched_x = batching(tensor_x)
batched_y = batching(tensor_y)

print(batched_x, batched_y)
print(len(batched_x), len(batched_y))

[tensor([[5, 8, 2],
        [8, 2, 9]]), tensor([[2, 9, 4]])] [tensor([[8, 2, 9],
        [2, 9, 4]]), tensor([[9, 4, 7]])]
2 2


# Embeddings

In [1146]:
num_embeddings = 10
num_dimensions = 4

embeddings = torch.nn.Embedding(num_embeddings, num_dimensions)

In [1147]:
token_vectors = embeddings(batched_x[0])

print(token_vectors)
print("Tabela:", embeddings.weight.shape)
print("Entrada:", batched_x[0].shape)
print("Resultado:", token_vectors.shape)
print(embeddings.weight[8])

tensor([[[-0.3142, -0.2511, -0.1658,  1.1284],
         [-1.2899, -0.9083,  0.8273,  0.9243],
         [ 1.2991,  0.3748,  0.7445, -1.5084]],

        [[-1.2899, -0.9083,  0.8273,  0.9243],
         [ 1.2991,  0.3748,  0.7445, -1.5084],
         [-0.3394,  0.9885, -2.4955, -1.2671]]], grad_fn=<EmbeddingBackward0>)
Tabela: torch.Size([10, 4])
Entrada: torch.Size([2, 3])
Resultado: torch.Size([2, 3, 4])
tensor([-1.2899, -0.9083,  0.8273,  0.9243], grad_fn=<SelectBackward0>)


# Posição

In [1148]:
num_linhas = 3
num_componentes = 4

posicao = torch.nn.Embedding(num_linhas, num_componentes)
value = torch.arange(num_linhas)

In [1149]:
position_vectors = posicao(value)
print(position_vectors, position_vectors.shape)

tensor([[-0.5040, -0.0860,  0.4872, -0.5278],
        [-0.4081,  2.6128,  0.4541,  0.2934],
        [-1.7746, -0.4366, -0.1711,  0.1868]], grad_fn=<EmbeddingBackward0>) torch.Size([3, 4])


In [1150]:
input_vectors = position_vectors + token_vectors
print(input_vectors, input_vectors.shape)
print("Tokens:", token_vectors)
print("Posições:", position_vectors)

tensor([[[-0.8182, -0.3371,  0.3214,  0.6006],
         [-1.6980,  1.7045,  1.2814,  1.2176],
         [-0.4756, -0.0618,  0.5734, -1.3216]],

        [[-1.7939, -0.9943,  1.3145,  0.3965],
         [ 0.8910,  2.9876,  1.1986, -1.2151],
         [-2.1140,  0.5519, -2.6666, -1.0803]]], grad_fn=<AddBackward0>) torch.Size([2, 3, 4])
Tokens: tensor([[[-0.3142, -0.2511, -0.1658,  1.1284],
         [-1.2899, -0.9083,  0.8273,  0.9243],
         [ 1.2991,  0.3748,  0.7445, -1.5084]],

        [[-1.2899, -0.9083,  0.8273,  0.9243],
         [ 1.2991,  0.3748,  0.7445, -1.5084],
         [-0.3394,  0.9885, -2.4955, -1.2671]]], grad_fn=<EmbeddingBackward0>)
Posições: tensor([[-0.5040, -0.0860,  0.4872, -0.5278],
        [-0.4081,  2.6128,  0.4541,  0.2934],
        [-1.7746, -0.4366, -0.1711,  0.1868]], grad_fn=<EmbeddingBackward0>)


# Atenção

In [1151]:
camadas_lineares = [
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
]

chamadas = []

for linear in camadas_lineares:
    chamadas.append(linear(input_vectors))

queries, keys, values = chamadas
chamadas = []

print(queries, keys, values)

tensor([[[ 0.2188,  0.5161,  0.2423,  0.5034],
         [ 0.7328,  0.1177,  1.2688,  1.1924],
         [-0.2314, -0.3697, -0.4286, -0.0753]],

        [[ 0.0574,  0.6699,  0.1147,  1.0847],
         [-0.3502, -2.1510,  0.3823, -0.1797],
         [ 1.8523,  1.8414,  0.0888, -1.0823]]], grad_fn=<UnsafeViewBackward0>) tensor([[[ 0.1004, -0.1980,  0.4344, -0.3623],
         [-0.0947, -0.8364, -0.1647, -1.3678],
         [ 0.6408,  0.0351,  0.5057,  0.1820]],

        [[ 0.6485, -0.1860,  1.2610, -0.4323],
         [-0.1461, -0.0227, -1.5001, -0.0447],
         [ 0.4250, -1.4711,  0.7749, -1.1872]]], grad_fn=<UnsafeViewBackward0>) tensor([[[ 0.0157,  0.3124, -0.1495, -0.0105],
         [ 0.4437, -0.0542, -0.7046,  0.9793],
         [-0.2538, -0.4399, -0.0376, -0.4582]],

        [[-0.0526,  0.2167, -0.3942, -0.3477],
         [ 0.5238, -1.6431, -0.4804,  1.0415],
         [-1.2192,  1.5070,  0.8498, -0.8059]]], grad_fn=<UnsafeViewBackward0>)


In [1152]:
resultado_produto_escalar = []
for i in range(3):
    resultado_produto_escalar.append(sum(queries[0, 1] * keys[0, i]))
    
tensor_pe = torch.stack(resultado_produto_escalar)
half_tensor_pe = tensor_pe/2
print(resultado_produto_escalar)
print(tensor_pe)
print(half_tensor_pe)


[tensor(0.1694, grad_fn=<AddBackward0>), tensor(-2.0078, grad_fn=<AddBackward0>), tensor(1.3324, grad_fn=<AddBackward0>)]
tensor([ 0.1694, -2.0078,  1.3324], grad_fn=<StackBackward0>)
tensor([ 0.0847, -1.0039,  0.6662], grad_fn=<DivBackward0>)


In [1153]:
mascara = half_tensor_pe.clone()
mascara[2] = float("-inf")
print(mascara)

tensor([ 0.0847, -1.0039,    -inf], grad_fn=<CopySlices>)


In [1154]:
attention_weights = torch.softmax(mascara, dim=0)
print(attention_weights)
print(attention_weights.sum())

tensor([0.7481, 0.2519, 0.0000], grad_fn=<SoftmaxBackward0>)
tensor(1., grad_fn=<SumBackward0>)


In [1155]:
weighted_vectors = []
for i in range(3):
    weighted_vectors.append(attention_weights[i]*values[0, i])
    
weighted_vectors = torch.stack(weighted_vectors)
print(weighted_vectors)
print(weighted_vectors.sum(dim=0))

tensor([[ 0.0117,  0.2337, -0.1118, -0.0079],
        [ 0.1118, -0.0136, -0.1775,  0.2467],
        [-0.0000, -0.0000, -0.0000, -0.0000]], grad_fn=<StackBackward0>)
tensor([ 0.1235,  0.2200, -0.2893,  0.2388], grad_fn=<SumBackward1>)


# Implementando atenção

In [1156]:
chamadas = []
camadas_lineares = [
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
    torch.nn.Linear(4, 4, bias=False),
]

for linear in camadas_lineares:
    chamadas.append(linear(input_vectors))

queries, keys, values = chamadas

In [1157]:
# deprecated
def calcular_pontuacoes_test(queries, keys):
    save3 = []

    for i in range(len(queries)):
        save2 = []
        for j in range(len(queries[i])):
            save = []
            for k in range(len(queries[i])):
                save.append(sum(queries[i, j] * keys[i, k]) / 2)

            save2.append(torch.stack(save))

        save3.append(torch.stack(save2))

    return torch.stack(save3)

In [1158]:
# New
def calcular_pontuacoes(queries, keys):
    dimensao = keys.shape[-1] ** 0.5
    transposed_keys = keys.transpose(-2, -1)
    return (queries @ transposed_keys) / dimensao

In [1159]:
# deprecated
def sliding_mascara(tensors):
    mascaras = []
    for index in range(len(tensors)):
        print(tensors[index])
        mascara = tensors[index].clone()
        mascara[index + 1 : len(mascara)] = float("-inf")
        mascaras.append(mascara)

    return torch.stack(mascaras)

In [1160]:
# new
def aplicar_mascara_causal(pontuacoes):
    pontuacoes_mascara = pontuacoes.clone()
    for exemplos_index in range(len(pontuacoes_mascara)):
        tokens_tamanho = len(pontuacoes_mascara[exemplos_index])
        for tokens_index in range(tokens_tamanho):
            pontuacoes_mascara[exemplos_index][tokens_index][tokens_index + 1 : tokens_tamanho] = float("-inf")

    return pontuacoes_mascara


In [1161]:
def calcular_pesos(pontuacoes_mascaradas):
    return torch.softmax(pontuacoes_mascaradas, dim=-1)

In [1162]:
def combinar_values(pesos, values):
    return pesos @ values

In [1163]:
def atencao_causal(queries, keys, values):
    tensor_pe = calcular_pontuacoes(queries, keys)
    pontuacoes_mascaradas = aplicar_mascara_causal(tensor_pe)
    pesos = calcular_pesos(pontuacoes_mascaradas)
    contextos = combinar_values(pesos, values)
    return pesos, contextos

In [1164]:
pesos, contextos = atencao_causal(queries, keys, values)

# Testes

In [1165]:
test2 = pesos @ values
print(test2.shape)
print(torch.allclose(contextos, test2))

torch.Size([2, 3, 4])
True


# Validação se altera o futuro

In [1166]:
queries_clone = queries.clone()
keys_clone = keys.clone()
values_clone = values.clone()

queries_clone[0][2] += 10
keys_clone[0][2] += 10
values_clone[0][2] += 10

In [1167]:
pesos_clone, contexto_clone = atencao_causal(queries_clone, keys_clone, values_clone)
print(torch.allclose(contexto_clone[0][0], contextos[0][0]))
print(torch.allclose(contexto_clone[0][1], contextos[0][1]))


True
True


# Melhorando atencao

In [1168]:
class AtencaoCausal(torch.nn.Module):
    def __init__(self, dimensao_entrada, dimensao_saida=0):
        super().__init__()
        self.dimensao_entrada = dimensao_entrada
        self.dimensao_saida = (
            dimensao_entrada if dimensao_saida <= 0 else dimensao_saida
        )
        self.camada_linear_query = torch.nn.Linear(
            dimensao_entrada, self.dimensao_saida, bias=False
        )
        self.camada_linear_key = torch.nn.Linear(
            dimensao_entrada, self.dimensao_saida, bias=False
        )
        self.camada_linear_value = torch.nn.Linear(
            dimensao_entrada, self.dimensao_saida, bias=False
        )

    def forward(self, x):
        queries = self.camada_linear_query(x)
        keys = self.camada_linear_key(x)
        values = self.camada_linear_value(x)
        return atencao_causal(queries, keys, values)

In [1169]:
atencao_cabeça_1 = AtencaoCausal(4, 2)
atencao_cabeça_2 = AtencaoCausal(4, 2)


In [1170]:
peso_atencao_1, contexto_atencao_1 = atencao_cabeça_1(input_vectors)
print(peso_atencao_1, contexto_atencao_1)
peso_atencao_2, contexto_atencao_2 = atencao_cabeça_2(input_vectors)
print(peso_atencao_2, contexto_atencao_2)

tensor([[[1.0000, 0.0000, 0.0000],
         [0.4927, 0.5073, 0.0000],
         [0.3306, 0.2346, 0.4348]],

        [[1.0000, 0.0000, 0.0000],
         [0.4034, 0.5966, 0.0000],
         [0.2911, 0.4364, 0.2725]]], grad_fn=<SoftmaxBackward0>) tensor([[[ 0.3278, -0.5536],
         [ 0.7486, -0.7383],
         [ 0.3990, -0.4456]],

        [[ 0.6686, -1.3508],
         [ 0.4260, -0.2096],
         [ 0.3016,  0.1436]]], grad_fn=<UnsafeViewBackward0>)
tensor([[[1.0000, 0.0000, 0.0000],
         [0.6132, 0.3868, 0.0000],
         [0.2699, 0.3600, 0.3701]],

        [[1.0000, 0.0000, 0.0000],
         [0.7890, 0.2110, 0.0000],
         [0.1989, 0.7027, 0.0984]]], grad_fn=<SoftmaxBackward0>) tensor([[[ 0.6281, -0.1566],
         [ 0.6629, -0.0823],
         [ 0.3678, -0.3363]],

        [[ 1.1703, -1.0166],
         [ 0.6131, -0.8067],
         [-0.7562, -0.1581]]], grad_fn=<UnsafeViewBackward0>)


In [1171]:
contexto_cabecas = torch.cat(
    (contexto_atencao_1, contexto_atencao_2), dim=2
)

projecao_saida = torch.nn.Linear(4, 4, bias=False)
contexto_projetado = projecao_saida(contexto_cabecas)

print(contexto_projetado)
print(contexto_projetado.shape)

tensor([[[-0.3309, -0.0627, -0.2061, -0.2248],
         [-0.4753, -0.2590, -0.3539, -0.0552],
         [-0.2417, -0.1900, -0.3422, -0.1748]],

        [[-0.5626, -0.2983, -0.8490, -0.7282],
         [-0.3882, -0.2020, -0.4500, -0.5318],
         [ 0.2371, -0.3651, -0.3021,  0.3673]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 3, 4])


# Atenção de multiplas cabeças

In [1172]:
class AtencaoMultiplasCabecas(torch.nn.Module):
    def __init__(self, dimensao_entrada, dimensao_saida=0, qtd_cabecas=1):
        super().__init__()
        self.dimensao_entrada = dimensao_entrada

        self.dimensao_saida = (
            dimensao_entrada if dimensao_saida <= 0 else dimensao_saida
        )

        if qtd_cabecas <= 0:
            raise ValueError(
                "A quantidade de cabeças precisa ser maior que zero."
            )

        if self.dimensao_saida % qtd_cabecas != 0:
            raise ValueError(
                "A dimensão de saída precisa ser divisível pela quantidade de cabeças."
            )

        self.qtd_cabecas = qtd_cabecas

        self.cabecas = torch.nn.ModuleList(
            [
                AtencaoCausal(
                    dimensao_entrada,
                    self.dimensao_saida // self.qtd_cabecas,
                )
                for _ in range(self.qtd_cabecas)
            ]
        )

        self.saida_linear = torch.nn.Linear(
            self.dimensao_saida,
            self.dimensao_saida,
            bias=False,
        )

    def forward(self, x):
        cabecas_atencao_contextos = []

        for index in range(len(self.cabecas)):
            cabecas_atencao_contextos.append(
                self.cabecas[index](x)[1]
            )

        cabecas_atencao = torch.cat(
            cabecas_atencao_contextos, dim=2
        )

        return self.saida_linear(cabecas_atencao)

In [1173]:
multi_cabecas = AtencaoMultiplasCabecas(4, 4, 2)
saida_multi_cabecas = multi_cabecas(input_vectors)

print(saida_multi_cabecas)
print(saida_multi_cabecas.shape)

clone_input_vector = input_vectors.clone()
clone_input_vector[0, 2] += 10

saida_multi_cabecas_clone = multi_cabecas(clone_input_vector)

print(
    torch.allclose(
        saida_multi_cabecas_clone[0, 0],
        saida_multi_cabecas[0, 0],
    )
)

print(
    torch.allclose(
        saida_multi_cabecas_clone[0, 1],
        saida_multi_cabecas[0, 1],
    )
)

tensor([[[-0.0732, -0.2574, -0.3743, -0.1215],
         [-0.0921, -0.2479, -0.9002, -0.3437],
         [-0.0859, -0.1487, -0.3115, -0.1894]],

        [[-0.2754, -0.6068, -0.5563, -0.3360],
         [-0.0491,  0.2101, -0.4183, -0.4701],
         [-0.1150,  0.0912, -0.2960, -0.3595]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 3, 4])
True
True


# Normalização

In [1174]:
saida_residual = input_vectors + saida_multi_cabecas

print(saida_residual)
print(saida_residual.shape)

tensor([[[-0.8914, -0.5946, -0.0529,  0.4791],
         [-1.7901,  1.4567,  0.3812,  0.8739],
         [-0.5615, -0.2106,  0.2619, -1.5110]],

        [[-2.0693, -1.6011,  0.7582,  0.0605],
         [ 0.8419,  3.1977,  0.7803, -1.6851],
         [-2.2290,  0.6431, -2.9626, -1.4398]]], grad_fn=<AddBackward0>)
torch.Size([2, 3, 4])


In [1175]:
vetor_teste = torch.tensor([1.0, 3.0, 5.0, 7.0])

media = vetor_teste.mean()
var = vetor_teste.var(unbiased=False)
normalizacao = (vetor_teste - media) / (var + 1e-5) ** 0.5

print("Vetor: ", vetor_teste)
print("Média: ", media)
print("Variância: ", var)
print("Vetor normalizado: ", normalizacao)

Vetor:  tensor([1., 3., 5., 7.])
Média:  tensor(4.)
Variância:  tensor(5.)
Vetor normalizado:  tensor([-1.3416, -0.4472,  0.4472,  1.3416])


In [1176]:
def normalizacao(vetor):
    media = vetor.mean(dim=2, keepdim=True)
    var = vetor.var(unbiased=False, dim=2, keepdim=True)
    normalizacao = (vetor - media) / (var + 1e-5) ** 0.5
    return normalizacao

In [1177]:
print(normalizacao(saida_residual))

tensor([[[-1.1948, -0.6286,  0.4045,  1.4190],
         [-1.6466,  0.9993,  0.1229,  0.5244],
         [-0.0864,  0.4534,  1.1803, -1.5472]],

        [[-1.1683, -0.7651,  1.2672,  0.6662],
         [ 0.0337,  1.3980, -0.0020, -1.4298],
         [-0.5430,  1.5878, -1.0873,  0.0425]]], grad_fn=<DivBackward0>)


# LayerNorm

In [1178]:
class LayerNorm(torch.nn.Module):
    def __init__(self, qtd_componentes):
        super().__init__()
        self.qtd_componentes = qtd_componentes
        self.epsilon = 1e-5
        self.vetor_escala = torch.nn.Parameter(torch.ones(qtd_componentes))
        self.vetor_deslocamento = torch.nn.Parameter(torch.zeros(qtd_componentes))

    def forward(self, x):
        normalizado = self.normalizacao(x)
        resultado = self.vetor_escala * normalizado + self.vetor_deslocamento
        return resultado

    def normalizacao(self, vetor, eixo=-1):
        media = vetor.mean(dim=eixo, keepdim=True)
        var = vetor.var(unbiased=False, dim=eixo, keepdim=True)
        normalizacao = (vetor - media) / (var + self.epsilon) ** 0.5
        return normalizacao

In [1179]:
layer = LayerNorm(4)
layer_saida_residual = layer(saida_residual)
print(layer_saida_residual)

tensor([[[-1.1948, -0.6286,  0.4045,  1.4190],
         [-1.6466,  0.9993,  0.1229,  0.5244],
         [-0.0864,  0.4534,  1.1803, -1.5472]],

        [[-1.1683, -0.7651,  1.2672,  0.6662],
         [ 0.0337,  1.3980, -0.0020, -1.4298],
         [-0.5430,  1.5878, -1.0873,  0.0425]]], grad_fn=<AddBackward0>)


# MLP

In [1180]:
class MLP(torch.nn.Module):
    def __init__(self, dimensao):
        super().__init__()
        self.dimensao = dimensao
        self.expansao = torch.nn.Linear(dimensao, 4 * dimensao)
        self.ativacao = torch.nn.GELU()
        self.projecao_saida = torch.nn.Linear(4 * dimensao, dimensao)

    def forward(self, x):
        expansao_vetor = self.expansao(x)
        ativacao_vetor = self.ativacao(expansao_vetor)
        projecao_saida_vetor = self.projecao_saida(ativacao_vetor)
        return projecao_saida_vetor

In [1181]:
mlp = MLP(4)
mlp_saida_residual = mlp(layer_saida_residual)

In [1182]:
conexao_residual_mlp = saida_residual + mlp_saida_residual

# print(layer_saida_residual)
# print(mlp_saida_residual)
print(conexao_residual_mlp)

tensor([[[-0.8387, -0.8123, -0.0726,  0.8330],
         [-1.8463,  1.4677,  0.4077,  1.1544],
         [-0.6891, -0.3148,  0.4825, -0.8090]],

        [[-1.9768, -1.9169,  0.8023,  0.6958],
         [ 0.8341,  3.3458,  0.7252, -1.4212],
         [-2.2180,  0.8387, -3.1115, -1.3612]]], grad_fn=<AddBackward0>)


In [1183]:
class BlocoTransformer(torch.nn.Module):
    def __init__(self, dimensao, qtd_cabecas):
        super().__init__()
        self.norm_atencao = LayerNorm(dimensao)
        self.atencao = AtencaoMultiplasCabecas(dimensao,dimensao, qtd_cabecas)
        self.norm_mlp = LayerNorm(dimensao)
        self.mlp = MLP(dimensao)
        
    def forward(self, x):
        entrada = self.norm_atencao(x)
        entrada_normalizada = self.atencao(entrada)
        residual_atencao = x + entrada_normalizada
        residual_normalizada = self.norm_mlp(residual_atencao)
        saida_mlp = self.mlp(residual_normalizada)
        saida_bloco = residual_atencao + saida_mlp
        return saida_bloco

In [1184]:
bloco_transformer = BlocoTransformer(4, 2)
saida_bloco = bloco_transformer(input_vectors)

In [1185]:
camada_linear = torch.nn.Linear(4, 10, bias=False)
logits = camada_linear(saida_bloco)

In [1186]:
print(torch.argmax(logits, dim=-1))
print(batched_y[0])

tensor([[3, 7, 0],
        [3, 6, 0]])
tensor([[8, 2, 9],
        [2, 9, 4]])


In [1187]:
softmax_logits = torch.softmax(logits[0][0], dim=0)
log = -torch.log(softmax_logits[8])
print(log)

tensor(2.6287, grad_fn=<NegBackward0>)


In [1188]:
save_error = []
choosen_batch = batched_y[0]
for i in range(len(choosen_batch)):
    save_error_example = []
    for j in range(len(choosen_batch[i])):
        softmax_logits = torch.softmax(logits[i][j], dim=0)
        save_error_example.append(-torch.log(softmax_logits[choosen_batch[i, j]]))

    save_error.append(torch.stack(save_error_example))

resultado = torch.stack(save_error)
print(resultado)
print(resultado.mean())

tensor([[2.6287, 2.3458, 2.3762],
        [2.4797, 2.4170, 1.7775]], grad_fn=<StackBackward0>)
tensor(2.3375, grad_fn=<MeanBackward0>)


In [1189]:
logits_entropia = logits.reshape(-1, logits.shape[-1])
batch_entropia = batched_y[0].reshape(-1)
print(logits_entropia)
print(batch_entropia)

tensor([[ 0.0549, -0.0575,  0.0872,  0.3349,  0.2460,  0.0748, -0.1100, -0.0789,
         -0.2802,  0.0467],
        [ 0.7633, -0.8882,  0.2478, -0.2443, -0.2267, -0.3493,  0.1532,  1.3815,
         -0.7991,  0.5060],
        [ 0.7337, -0.6472, -0.1485,  0.2389,  0.3626, -0.8294,  0.4557, -0.2515,
          0.2063,  0.0433],
        [ 0.1663, -0.2174, -0.0180,  0.9085,  0.3357,  0.2920, -0.5201,  0.4224,
         -1.1098,  0.0855],
        [ 1.1614, -1.1332, -0.1596, -1.3819, -0.6399, -1.5452,  1.2245,  0.7977,
          0.9855,  0.2910],
        [ 1.9835, -1.8092,  0.7772,  0.1221,  1.4410, -2.1174,  1.6932, -1.1274,
          0.7643,  0.6934]], grad_fn=<ViewBackward0>)
tensor([8, 2, 9, 2, 9, 4])


In [1190]:
erro = torch.nn.functional.cross_entropy(logits_entropia, batch_entropia)
erro.backward()

In [1191]:
print(camada_linear.weight.grad)
print(camada_linear.weight.grad.shape)

tensor([[-0.2118,  0.1112, -0.0863, -0.1473],
        [-0.0460, -0.0016,  0.0117, -0.0106],
        [ 0.4589, -0.0353, -0.3054, -0.1990],
        [-0.1252, -0.0238,  0.0164, -0.0274],
        [ 0.2307, -0.0095,  0.4408,  0.1898],
        [-0.0649, -0.0082,  0.0213, -0.0019],
        [-0.1409,  0.1105, -0.0667, -0.1313],
        [-0.1352,  0.1054,  0.0692, -0.0118],
        [ 0.0266,  0.1593,  0.0059, -0.1058],
        [ 0.0077, -0.4079, -0.1069,  0.4454]])
torch.Size([10, 4])


In [1192]:
sgd = torch.optim.SGD(camada_linear.parameters(), lr=0.1)
pesos_antes = camada_linear.weight.detach().clone()

sgd.step()

print(torch.allclose(pesos_antes, camada_linear.weight))

False


In [1193]:
logits_2 = camada_linear(saida_bloco)
logits_entropia_2 = logits_2.reshape(-1, logits_2.shape[-1])
erro2 = torch.nn.functional.cross_entropy(logits_entropia_2, batch_entropia)

In [1194]:
print(torch.allclose(erro, erro2))
print(erro, erro2)

False
tensor(2.3375, grad_fn=<NllLossBackward0>) tensor(2.2152, grad_fn=<NllLossBackward0>)


In [1195]:
class GptPequeno(torch.nn.Module):
    def __init__(self, tamanho_vocabulario, context_size, dimensao, qtd_cabecas):
        super().__init__()
        self.embedding_tokens = torch.nn.Embedding(tamanho_vocabulario, dimensao)
        self.embedding_posicoes = torch.nn.Embedding(context_size, dimensao)
        self.transformer = BlocoTransformer(dimensao, qtd_cabecas)
        self.layer_norm = LayerNorm(dimensao)
        self.projecao_logits = torch.nn.Linear(dimensao, tamanho_vocabulario, bias=False)
    
    def forward(self, x):
        embeddings_tokens = self.embedding_tokens(x)
        indices_posicoes = torch.arange(0, x.shape[1], device=x.device)
        embeddings_posicoes = self.embedding_posicoes(indices_posicoes)
        entrada_transformer = embeddings_tokens + embeddings_posicoes
        saida_transformer = self.transformer(entrada_transformer)
        saida_normalizada = self.layer_norm(saida_transformer)
        logits = self.projecao_logits(saida_normalizada)
        return logits

In [1196]:
gpt = GptPequeno(10,3,4,2)
gpt_resultado = gpt(batched_x[0])
print(gpt_resultado)
print(gpt_resultado.shape)


tensor([[[ 0.4692, -0.1444, -0.1946, -0.9100, -0.1059,  0.3712,  0.5170,
          -0.1580, -0.0834,  0.0021],
         [ 0.3585, -0.0984, -0.2571, -1.0602,  0.2765,  0.5856,  0.2601,
          -0.1160, -0.6237, -0.3455],
         [-0.5612,  0.0562,  0.1550,  0.9201, -0.0054, -0.4512, -0.5305,
           0.1514,  0.3950,  0.0428]],

        [[ 0.7206, -0.0021, -0.0215, -0.6509, -0.2996,  0.2454,  0.7697,
          -0.1728, -0.0902,  0.3024],
         [-0.9902, -0.3343, -0.4307, -0.3656,  0.7242,  0.2508, -1.0576,
           0.1303, -0.0923, -0.9767],
         [-0.6995, -0.1554, -0.0221,  0.6076, -0.0710, -0.4084, -0.5612,
           0.1163,  0.7686, -0.0500]]], grad_fn=<UnsafeViewBackward0>)
torch.Size([2, 3, 10])


In [1197]:
gpt_sgd = torch.optim.SGD(gpt.parameters(), lr=0.1)
gpt_sgd.zero_grad()

In [1198]:
gpt_resultado_reshape = gpt_resultado.reshape(-1, gpt_resultado.shape[-1])
gpt_erro = torch.nn.functional.cross_entropy(gpt_resultado_reshape, batch_entropia)
gpt_erro_save = gpt_erro.detach().clone()

In [1199]:
gpt_erro.backward()
gpt_sgd.step()

In [1200]:
print(gpt_erro, gpt_erro_save)

tensor(2.5689, grad_fn=<NllLossBackward0>) tensor(2.5689)


In [1201]:
gpt_resultado_2 = gpt(batched_x[0])
print(gpt_resultado)
print(gpt_resultado_2)

tensor([[[ 0.4692, -0.1444, -0.1946, -0.9100, -0.1059,  0.3712,  0.5170,
          -0.1580, -0.0834,  0.0021],
         [ 0.3585, -0.0984, -0.2571, -1.0602,  0.2765,  0.5856,  0.2601,
          -0.1160, -0.6237, -0.3455],
         [-0.5612,  0.0562,  0.1550,  0.9201, -0.0054, -0.4512, -0.5305,
           0.1514,  0.3950,  0.0428]],

        [[ 0.7206, -0.0021, -0.0215, -0.6509, -0.2996,  0.2454,  0.7697,
          -0.1728, -0.0902,  0.3024],
         [-0.9902, -0.3343, -0.4307, -0.3656,  0.7242,  0.2508, -1.0576,
           0.1303, -0.0923, -0.9767],
         [-0.6995, -0.1554, -0.0221,  0.6076, -0.0710, -0.4084, -0.5612,
           0.1163,  0.7686, -0.0500]]], grad_fn=<UnsafeViewBackward0>)
tensor([[[ 0.4397, -0.1715, -0.0638, -0.8732, -0.1702,  0.3329,  0.4867,
          -0.1802,  0.0280, -0.0955],
         [ 0.3436, -0.1226, -0.1206, -1.0269,  0.1969,  0.5521,  0.2395,
          -0.1427, -0.5367, -0.4288],
         [-0.5275,  0.0733,  0.0292,  0.9034,  0.0447, -0.4355, -0.4913,
    

In [1202]:
gpt_resultado_reshape_2 = gpt_resultado_2.reshape(-1, gpt_resultado_2.shape[-1])
gpt_erro_2 = torch.nn.functional.cross_entropy(gpt_resultado_reshape_2, batch_entropia)

In [1203]:
print(gpt_erro_2, gpt_erro_save)

tensor(2.4434, grad_fn=<NllLossBackward0>) tensor(2.5689)


In [ ]:
gpt = GptPequeno(10,3,4,2)
gpt_sgd = torch.optim.SGD(gpt.parameters(), lr=0.1)

In [1208]:
for i in range(100):
    gpt_sgd.zero_grad()
    gpt_resultado = gpt(batched_x[0])
    gpt_resultado_reshape = gpt_resultado.reshape(-1, gpt_resultado.shape[-1])
    gpt_erro = torch.nn.functional.cross_entropy(gpt_resultado_reshape, batch_entropia)
    gpt_erro.backward()
    gpt_sgd.step()
    
    if(i % 9 == 0):
        print(gpt_erro.item())

0.014000310562551022
0.013091925531625748
0.012283257208764553
0.011559401638805866
0.01090830098837614
0.01031985878944397
0.009785893373191357
0.009299500845372677
0.008854787796735764
0.008446821011602879
0.0080714737996459
0.007725119590759277


In [1218]:
gpt_resultado = gpt(batched_x[0][0:1])
logits_ultima_posicao = gpt_resultado[:,-1,:]
print(logits_ultima_posicao)
print(torch.argmax(logits_ultima_posicao, dim=-1, keepdim=True))
print(batched_y[0])

tensor([[ 0.2694,  0.5099, -7.9313,  0.9488,  3.6415, -0.1371,  0.2770,  0.3563,
         -4.4187,  8.9820]], grad_fn=<SelectBackward0>)
tensor([[9]])
tensor([[8, 2, 9],
        [2, 9, 4]])
